In [1]:
import sys
import os

# Add the parent directory (src) to the system path
# The '..' tells it to look one folder up from where the notebook is currently running
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
from ingest import load_faq_data

documents = load_faq_data()

In [3]:
documents_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]

In [4]:
len(documents_llm)

103

In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
#Build the insruction of  LLM for document generation
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from client import client

In [8]:
documents = documents_llm
doc = documents[0]

In [9]:
import json

user_prompt = json.dumps(doc)
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [10]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [11]:
response = client.responses.parse(
    model="gpt-4o-mini",
    input=messages,
    text_format=Questions
)

In [12]:
result = response.output_parsed
print(result)

questions=['Is it too late for me to enroll in this course since I just found out about it?', 'If I join the course now, will I still be able to get a certificate?', 'What do I need to do for certificate eligibility if I join after the course has started?', 'Can I still submit my project later if I start the course now?', 'What is the deadline for project submission to qualify for the certificate?']


### Combining in modular form

In [13]:
from evaluation_utils import llm_structured

In [14]:
result, usage = llm_structured(
    client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Is it too late to join the LLM Zoomcamp course?', 'Can I get a certificate if I join the course now?', 'What do I need to do to receive a certificate in this course?', 'Are there any deadlines I should know about for submitting my project?', 'Can I still take the course even though I found it late?']


In [15]:
usage

ResponseUsage(input_tokens=211, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=75, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=286, cost=7.665e-05, is_byok=False, cost_details={'upstream_inference_cost': 7.665e-05, 'upstream_inference_input_cost': 3.165e-05, 'upstream_inference_output_cost': 4.5e-05})

In [16]:
from evaluation_utils import calc_price

cost = calc_price(usage)
cost

{'input_cost': 0.00015825,
 'output_cost': 0.00033749999999999996,
 'total_cost': 0.00049575}

In [17]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Is it too late to join the LLM Zoomcamp course?',
  'document': '74eb249bbf'},
 {'question': 'Can I get a certificate if I join the course now?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to receive a certificate in this course?',
  'document': '74eb249bbf'},
 {'question': 'Are there any deadlines I should know about for submitting my project?',
  'document': '74eb249bbf'},
 {'question': 'Can I still take the course even though I found it late?',
  'document': '74eb249bbf'}]